# 15 — Pitfalls

**The concept:** this page is the one to read twice. Every item here produces a **wrong answer
rather than an error** — the pipeline runs, converges, and the arithmetic is right.

All of them are in `docs/known-issues.md` with detail.

## 1. Duplicate output names overwrite silently

Two steps declaring the same output: the last registered wins, **the first still runs**, its result
is discarded, and consumers wire to the wrong producer.

In [1]:
from smartmdao import Pipeline, validate

ran = []

duplicate = Pipeline()

@duplicate.step(outputs=["mass"])
def detailed_mass(span: float) -> float:
    ran.append("detailed_mass")
    return span * 12.0

@duplicate.step(outputs=["mass"])
def rough_mass(span: float) -> float:
    ran.append("rough_mass")
    return 999.0

result = duplicate.run(span=10.0)
print("mass  ->", result["mass"], " (from rough_mass, the last registered)")
print("ran   ->", ran, " <- BOTH executed; one result was thrown away")
print()
print("validate() catches it:", [f.code for f in validate(duplicate, inputs=["span"])])

mass  -> 999.0  (from rough_mass, the last registered)
ran   -> ['detailed_mass', 'rough_mass']  <- BOTH executed; one result was thrown away

validate() catches it: ['duplicate-output']


## 2. Which variable needs a seed depends on step *names*

`HybridSolver` sorts a cyclic block alphabetically. Rename a step and the required seed can move —
with no change to the physics.

In [2]:
from smartmdao import HybridSolver, analyze

def loop_with_names(first_name, second_name):
    p = Pipeline(solver=HybridSolver())
    def a(y2: float, z: float) -> float:
        return z**2 - 0.2 * y2
    def b(y1: float) -> float:
        return abs(y1) ** 0.5
    a.__name__, b.__name__ = first_name, second_name
    p.add(a, outputs=["y1"])
    p.add(b, outputs=["y2"])
    return p

for names in (("discipline_1", "discipline_2"), ("zulu", "alpha")):
    p = loop_with_names(*names)
    seeds = [g.variable for g in analyze(p, inputs=["z"]).initial_guesses_required]
    print(f"{str(names):34} -> seed {seeds}")

('discipline_1', 'discipline_2')   -> seed ['y2']
('zulu', 'alpha')                  -> seed ['y1']


## 3. A step returning `None` stores nothing

`None` is how a side-effect-only step is expressed, so it cannot also mean "no value". Inside a
loop, the **previous value persists** — and a convergence checker reads that as *at rest*.

In [3]:
from smartmdao import IterativeSolver

sneaky = Pipeline(solver=IterativeSolver(max_iterations=10, target_var="choice"))

@sneaky.step(outputs=["choice"])
def sometimes_undecided(load: float) -> str:
    return None            # "I have no answer" - but nothing is stored

out = sneaky.run(load=1.0, choice="initial")
print("choice ->", out["choice"], " <- the seed, never replaced")
print("status ->", out["convergence_reports"][-1].status, " <- reported as CONVERGED")
print()
print("A discipline must be TOTAL: return an explicit sentinel, never None.")

choice -> initial  <- the seed, never replaced
status -> converged  <- reported as CONVERGED

A discipline must be TOTAL: return an explicit sentinel, never None.


## 4. `IterativeSolver` ignores the dependency graph

It sweeps in **registration order**. Register a consumer before its producer and the first sweep
sees only the initial value — which can look like immediate convergence.

In [4]:
misordered = Pipeline(solver=IterativeSolver(max_iterations=20, target_var="architecture"))

@misordered.step(outputs=["architecture"])
def choose(requirements: str) -> str:      # registered FIRST, but depends on the next step
    return "compliant" if requirements == "strict" else "basic"

@misordered.step(outputs=["requirements"])
def derive(architecture: str) -> str:
    return "strict" if architecture == "basic" else "relaxed"

report = misordered.run(architecture="basic", requirements="strict")["convergence_reports"][-1]
print("status:    ", report.status)
print("iterations:", report.iterations)
print()
print("HybridSolver derives order from the graph and avoids this entirely.")

Reached max_iterations (20) without converging. Last residual: inf


status:     max_iterations
iterations: 20

HybridSolver derives order from the graph and avoids this entirely.


## 5. A disconnected discipline converges happily

The most expensive mistake here, and it was found in real use. Part of the pipeline cannot affect
the answer, and **everything looks fine**.

In [5]:
split = Pipeline(solver=HybridSolver())

@split.step(outputs=["lift"])
def compute_lift(span: float, chord: float, speed: float) -> float:
    return 0.5 * 1.225 * speed**2 * span * chord      # nothing reads `lift`

@split.step(outputs=["mass"])
def mass_loop(mass: float) -> float:
    return 100.0 + 0.05 * mass

out = split.run(span=10.0, chord=1.5, speed=50.0, mass=0.0)
print("converged:", out["convergence_reports"][0].status)
print("mass:     ", round(out["mass"], 6))
print()
print("Now change span, chord and speed by a factor of ten:")
out2 = split.run(span=100.0, chord=15.0, speed=500.0, mass=0.0)
print("mass:     ", round(out2["mass"], 6), " <- identical. Three inputs do nothing.")
print()
for finding in validate(split, inputs=["span", "chord", "speed", "mass"]):
    print(f"[{finding.code}] {finding.message[:110]}...")

converged: converged
mass:      105.263158

Now change span, chord and speed by a factor of ten:
mass:      105.263158  <- identical. Three inputs do nothing.

[disconnected-graph] The pipeline falls into 2 disconnected pieces, so nothing computed in one can affect another: ['compute_lift']...


## 6. `@cached` functions are keyword-only

Invisible inside a pipeline, surprising the moment you test a discipline on its own.

In [6]:
from smartmdao import cached, MemoryBackend

@cached(backend=MemoryBackend())
def discipline(a: float, b: float) -> float:
    return a + b

try:
    discipline(1.0, 2.0)
except TypeError as error:
    print(f"TypeError: {error}")
print("keywords work:", discipline(a=1.0, b=2.0))

TypeError: discipline() takes 0 positional arguments but 2 were given
keywords work: 3.0


## 7. A threshold inside a loop can give the loop two answers

Both converged, both self-consistent, and the initial guess alone decides which one you get. See
[11 — Discretisation](11-discretisation.ipynb).

In [7]:
from smartmdao import Bands, Discretisation

def bistable():
    p = Pipeline(solver=HybridSolver(),
                 discretisation=Discretisation(
                     mass_band=Bands("total_mass_kg", edges=[500], names=["light", "heavy"])))

    @p.step(outputs=["structure_mass_kg"])
    def size_structure(mass_band: str) -> float:
        return 200.0 if mass_band == "light" else 260.0

    @p.step(outputs=["total_mass_kg"])
    def sum_masses(structure_mass_kg: float, payload_kg: float) -> float:
        return structure_mass_kg + payload_kg

    return p

for seed in (400.0, 900.0):
    out = bistable().run(payload_kg=250.0, total_mass_kg=seed)
    print(f"seed {seed:6.1f} -> {out['mass_band']:6} at {out['total_mass_kg']:6.1f} kg "
          f"({out['convergence_reports'][0].status})")

seed  400.0 -> light  at  450.0 kg (converged)
seed  900.0 -> heavy  at  510.0 kg (converged)


## 8. `int` does not satisfy `float`

Strict by design. The answer is the documented extension point, not a hidden default — see
[6 — Type checking](06-type-checking.ipynb).

In [8]:
from smartmdao import StandardTypeChecker
print("int -> float:", StandardTypeChecker().check_types(int, float))

int -> float: False


## The habit that catches all of these

```python
analyze(pipeline, inputs=[...])     # order, cycles, which variable needs a seed
validate(pipeline, inputs=[...])    # every structural problem at once
```

Neither executes a discipline. Both are free. **Run them.**

---

Back to [the index](README.md).